<a href="https://colab.research.google.com/github/DeveshPandey1331/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -q duckdb datasets pyarrow

In [16]:
import duckdb
import pandas as pd
from google.colab import userdata

In [17]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [18]:
con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [19]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [20]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeveshPandey1331/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



One row represents the daily performance of a single content item for a single client on one report date.

Time Window:
This analysis uses data from a single mid-panel month (March 2026) to avoid using the final outcome month and to reduce leakage.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        report_date,
        client_hash_id,
        content_hash_id
    )) AS unique_rows
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,9841378,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

- gsc_impressions
- gsc_clicks
- gsc_sum_position
- client_has_gsc
- client_has_ga4

## Label

gsc_clicks

## Context

report_date
client_hash_id
content_hash_id
month

## Excluded

AI referral columns (ai_chatgpt, ai_claude, etc.) because they are not required for this simple baseline analysis.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
report_date,
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_sum_position,
client_has_gsc,
client_has_ga4
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 10
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,30,0,115,True,True
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,5,0,358,True,True
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,1,0,34,True,True
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,6,0,140,True,True
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,5,0,89,True,True
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e,21,0,1050,True,True
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,13,0,127,True,True
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,29,0,356,True,True
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602,5,0,103,True,True
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca,8,0,304,True,True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
COUNT(*) total_rows,
COUNT(DISTINCT (
report_date,
client_hash_id,
content_hash_id
))
AS unique_rows
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,9841378,9841378


In [24]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").df()

,row_count,first_day,last_day
0,9841378,2026-03-01,2026-03-31


In [25]:
con.sql(f"""
SELECT
COUNT(*) available_rows
FROM read_parquet(
'{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE
month='2026-03'
AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limitations

This dataset is an unbalanced panel, meaning different clients have different history lengths.

Some rows have only Google Search Console data or only GA4 data.

The analysis is restricted to one month and therefore does not capture long-term seasonal behaviour.

The dataset is pseudonymized, so client identities and page URLs cannot be interpreted directly.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.